# 03 · 고급 분석: 패널 고정효과  (모듈 4-A)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amnotyoung/oda-ai-stats/blob/main/notebooks/03_panel_fe.ipynb)

> 고급 계량 분석도 AI 도움으로 **양쪽 도구에서** 할 수 있다. 국가×연도 패널의 고정효과를
> Python으로, 그리고 STATA로 — **둘 다 같은 결과**. 도구는 우열이 아니라 **환경**으로 고른다.

In [ ]:
# [한 번 실행하고 넘어가세요] 한글 폰트 설정 — Colab 그래프의 한글이 깨지지 않도록(처음 1회 ~20초)
import os, matplotlib.pyplot as plt, matplotlib.font_manager as fm
try:
    p = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"   # 나눔고딕 폰트 경로
    if not os.path.exists(p):                                # 없으면
        os.system("apt-get -qq install -y fonts-nanum > /dev/null 2>&1")  # 설치
    fm.fontManager.addfont(p)                               # matplotlib에 폰트 등록
    plt.rc("font", family="NanumGothic")                   # 기본 글꼴로 지정
except Exception:
    pass                                                    # 실패해도 그냥 진행(영문은 정상)
plt.rc("axes", unicode_minus=False)                         # 마이너스 기호 깨짐 방지

## 0) 준비 — 패널 전용 라이브러리 설치
`linearmodels`는 고정효과를 전용으로 다룬다(STATA `xtreg`에 해당). Colab엔 기본 없어 한 번만 설치.

In [ ]:
# linearmodels = 패널(고정효과) 전용 라이브러리. Colab에 없으면 자동 설치(~20초)
try:
    import linearmodels
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "linearmodels"]); import linearmodels

In [ ]:
import pandas as pd, numpy as np
from linearmodels.panel import PooledOLS, PanelOLS   # 패널 회귀(합동 · 고정효과)
df = pd.read_csv("https://raw.githubusercontent.com/amnotyoung/oda-ai-stats/main/data/wdi_panel.csv").dropna(subset=["gdp_pc","life_exp"])  # 불러오며 핵심 결측 제거
df["log_gdp"] = np.log(df["gdp_pc"])                            # 로그변환
# 패널 구조 선언: (개체=국가, 시간=연도) 2단 인덱스를 주면 linearmodels가 패널로 인식
panel = df.set_index(["economy", "year"])
print("국가:", df["economy"].nunique(), "· 연도:", df["year"].nunique(), "· 행:", len(df))  # 패널 규모

## 문제 — 국가·시대의 교란을 통제하려면? (식별의 문제)
소득과 기대수명은 둘 다 **시간이 가며 함께 오른다.** 단순 상관엔 *국가 고유차*와 *전세계 시대추세*가
뒤섞여 있다. 진짜 효과를 보려면 이 둘을 **고정효과**로 걷어내야 한다 — 임팩트평가의 핵심 아이디어.

> **고정효과(FE)** = 국가/연도별 평균 차이를 통째로 빼는 것. `linearmodels`에선 `EntityEffects`(국가)·
> `TimeEffects`(연도)만 적으면 더미를 일일이 안 만들어도 흡수한다. (개념상 국가/연도 더미를 전부 넣는 것과 동일)

### (1) 통제 안 함 — 합동(pooled) 회귀

In [ ]:
# (1) 아무것도 통제 안 함 — 모든 국가·연도를 한 덩어리로 ("1 +"은 절편 항)
pooled = PooledOLS.from_formula("life_exp ~ 1 + log_gdp", panel).fit()
print("pooled         log_gdp =", round(pooled.params["log_gdp"], 2))

### (2) 국가 고정효과 — 국가 고유차 통제

In [ ]:
# (2) EntityEffects = 국가 고정효과(국가 고유차 흡수)
#     cluster_entity=True = 국가 단위 클러스터 표준오차(STATA vce(cluster)와 대응)
fe = PanelOLS.from_formula("life_exp ~ log_gdp + EntityEffects", panel)\
             .fit(cov_type="clustered", cluster_entity=True)
print("국가 FE        log_gdp =", round(fe.params["log_gdp"], 2), "(국가 고유효과 흡수)")

### (3) **이원 고정효과** — 국가 + 연도 동시 통제 (고급)
연도 고정효과(`TimeEffects`)까지 더하면 전세계 공통 **시대추세**가 빠지며 효과가 또 달라진다.

In [ ]:
# (3) + TimeEffects = 연도 고정효과 → 국가+연도 동시 통제(전세계 공통 시대추세 제거)
twfe = PanelOLS.from_formula("life_exp ~ log_gdp + EntityEffects + TimeEffects", panel)\
               .fit(cov_type="clustered", cluster_entity=True)
print("이원 FE(+연도) log_gdp =", round(twfe.params["log_gdp"], 2), "(시대추세까지 흡수)")
print("\n→ 4.59 → 3.54 → 1.26 : 무엇을 통제하느냐에 따라 답이 바뀐다(식별의 문제).")

> 🖼️ **그림으로**: 세 모형의 소득 계수를 **막대**로 — 통제를 더할수록 계수가 줄어드는 게 한눈에.

In [ ]:
# 세 모형의 '소득 계수'를 막대로 비교 — 통제를 더할수록 어떻게 변하나(식별의 문제)
import matplotlib.pyplot as plt
labels = ["통제 없음\n(pooled)", "국가 FE", "이원 FE\n(국가+연도)"]                 # 3개 모형
coefs  = [pooled.params["log_gdp"], fe.params["log_gdp"], twfe.params["log_gdp"]]  # 각 소득 계수
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(labels, coefs, color=["#bbbbbb", "#76a5af", "#4c78a8"])
for b, c in zip(bars, coefs):                                    # 막대 위에 계수 값 표기
    ax.text(b.get_x() + b.get_width() / 2, c, f"{c:.2f}", ha="center", va="bottom", fontweight="bold")
ax.set_ylabel("log_gdp 계수 (소득 효과)")
ax.set_title("통제를 더할수록 줄어드는 소득 효과 (4.59 → 3.54 → 1.26)")
plt.tight_layout(); plt.show()

### (4) STATA에선 — 같은 분석
```stata
encode economy, gen(country_id)
xtset country_id year
xtreg life_exp log_gdp i.year, fe vce(cluster country_id)
```
- `xtreg ..., fe`가 `PanelOLS`의 `EntityEffects`에, `i.year`가 `TimeEffects`에 대응. base STATA — **폐쇄망에서 인터넷 없이 실행** ✅  코드 → `stata/05_panel_fe.do`
- 두 도구 계수가 **정확히 일치**(1.262)한다 — 교차검증.

---
✅ **포인트**: 이원 고정효과는 **차분-차분(DiD) 임팩트평가의 엔진**이다(KOICA M&E와 직결).
"무엇을 통제했나"가 결과를 좌우하므로 *설계·검증은 사람 몫* — AI는 코드를 짜고, 판단은 여러분이 한다.